# 3D Fluorescence Nuclei Segmentation (v1.6.3)

Segments and quantifies nuclei and cell markers in 3D (or 2D) fluorescence images.

> Python 3.10–3.14. Install: `uv pip install -r requirements.txt --override overrides.txt`. On 3.14, StarDist and `.czi` files are unavailable.

**How to use**
1. Set file, ROI and experiment name in **Cell 3**, image processing and exports in **Cell 4**.
2. Run Cell 5, fill in the stain table and click **Confirm**.
3. Run the rest; if needed, tune Cell 4 and re-run from Cell 13.

**Methods:** watershed (default), Cellpose (`trig_cellpose`), StarDist (`trig_stardist`). Cell shape: nuclei-seeded watershed or Cellpose (`trig_cellpose_cyto`).

# 1. Import Libraries and Helpers

**Cell 1** — Import libraries (prints the Python version).

In [ ]:
from helpers.notebook_setup_helpers import load_nuclei_notebook_setup
globals().update(load_nuclei_notebook_setup())

**Cell 2** — Load helper functions. Re-run after editing `helpers/notebook_helpers.py`.

In [ ]:
from helpers.notebook_helpers import reload_helpers
reload_helpers()

# 2. Inputs and Setup

The only cells to edit for a new dataset.

**Cell 3** — Settings. A folder as `input_file` runs **batch mode**. Each setting is explained in line.

In [ ]:
# --- Input ---
input_file = 'path/to/your_image_file.nd2'  # file (.nd2/.tif/.czi/.lif/.oib) or folder (batch)
ROI = [0, 0, 0, 0, 0, 0]                    # [x0, x1, y0, y1, z0, z1]; 0 = full range
interactive_roi = False                     # True = draw ROI in napari

# --- Experiment ---
name_setup = 'your_experiment_name'  # prefix of saved files
use_setup = True                     # reuse saved stain table and contrast
automatic_contrast = False           # True = auto contrast, no napari window

# --- Sizes ---
nuclei_diameter = 10            # um
cell_diameter = 30              # um
scale_factor = 1.0
zoom_factors = [1.0, 1.0, 1.0]  # [Z, Y, X]

# --- Segmentation method (all False = watershed) ---
trig_cellpose = False       # Cellpose for nuclei (GPU recommended)
trig_stardist = False       # StarDist for nuclei (not on Python 3.14)
trig_cellpose_cyto = False  # Cellpose for cell shape (needs CYTOPLASM channel)

# --- Quantification ---
multilabel = True                 # multi-marker combinations in summary
aggregate_distance_factor = 0.05  # cells closer than this x cell_diameter = one aggregate
pcm_thickness_factor = 0.05       # PCM shell thickness = this x cell_diameter

# --- Nuclei size refinement ---
size_reference = "blend"  # "blend", "input" or "detected"
iterative_split = True    # re-split oversized nuclei
oversize_factor = 2.0     # oversized = > this x reference
max_split_iterations = 3  # 1-3 split levels
merge_undersized = True   # merge small fragments
undersize_factor = 0.5    # fragment = < this x reference

# --- Splitting ---
split_gamma = 1.5         # >1 = sharper seams between objects (1.0 = off)
split_gamma_step = 0.5    # gamma increase per re-split level (0 = off)
min_piece_fraction = 0.2  # merge split slivers < this x nucleus (0 = off)

**Cell 4** — Image processing and export settings, used in single-file and batch mode. To tune them, edit and run this cell, then re-run from Cell 13.

In [ ]:
# --- Image processing ---
# sigma and the median filter (Cell 11) work in voxels of the resampled image:
# at a higher scale_factor they act over a smaller distance in um.
sigma = 0.5                # smoothing (Cell 13): 0 = none, 0.5 = mild, 1-2 = strong (noisy data)
num_plateaus = 2           # contrast boost (Cell 14): raise to 3-4 for dim channels
plateau_factor = 0.7       # contrast boost limit (Cell 14): 0.5-1.0; lower = gentler
threshold_method = 'otsu'  # threshold (Cell 15): 'otsu' (clear), 'median' (sparse) or 'huang' (dim)

# --- 3D exports (skipped for 2D images) ---
export_vtk = True      # VTK volumes for ParaView (slow)
export_stl = True      # STL mesh per marker (slow)
export_fea = True      # FEA mesh, Abaqus .inp (slow)

**Cell 5** — Load image and stain table. Single-plane images are handled as 2D (`is_2d`).

Fill in the table: channel, color and marker per condition; tick **Intracellular?** or **Membrane?** for cytoplasm stains. Click **Confirm**.

In [ ]:
batch_mode = is_batch_input(input_file)
file_meta = None
is_2d = False  # set from the loaded image
stain_table_path = f"{name_setup}_stain_dict.json" if use_setup else None

if batch_mode:
    print(f"Batch mode: {len(list_image_files(input_file))} image(s) will be "
          f"processed by Cell 6.")
else:
    if interactive_roi:
        ROI = select_roi_interactively(input_file, ROI, napari_module=napari,
                                       stain_table_path=stain_table_path)
    (meta, img, r_X, r_Y, r_Z, file_meta, ROI_print,
     cyto_factor, PCM_factor, zooms, used_lazy_loading) = initialize_dataset(
        input_file, ROI,
        nuclei_diameter=nuclei_diameter,
        cell_diameter=cell_diameter,
        scale_factor=scale_factor,
        zoom_factors=zoom_factors,
    )
    is_2d = img.shape[0] == 1
    print("2D image: segmentation in 2D, sizes as areas [um2], 3D exports skipped."
          if is_2d else f"3D image: {img.shape[0]} Z planes.")

channel_list = print_channel_list(input_file, file_meta=file_meta)
stain_editor = edit_stain_dict(channel_list, name_setup=name_setup, use_saved=use_setup)

**Cell 6** — Read the stain table. In batch mode, processes the whole folder (to `<folder>_output/`) with the settings of Cells 3–4 and stops.

In [ ]:
stain_dict, cyto_markers = stain_editor.result()
membrane_markers = stain_editor.membrane_markers  # also in cyto_markers
print(f"stain_dict = {stain_dict}")
print(f"cyto_markers = {cyto_markers}")
print(f"membrane_markers = {membrane_markers}")

if batch_mode:
    batch_summary = run_batch_folder(
        input_file,
        roi_coords=ROI,
        stain_dict=stain_dict,
        cyto_markers=cyto_markers,
        membrane_markers=membrane_markers,
        name_setup=name_setup,
        use_setup=use_setup,
        automatic_contrast=automatic_contrast,
        interactive_roi=interactive_roi,
        nuclei_diameter=nuclei_diameter,
        cell_diameter=cell_diameter,
        scale_factor=scale_factor,
        zoom_factors=zoom_factors,
        trig_cellpose=trig_cellpose,
        trig_stardist=trig_stardist,
        trig_cellpose_cyto=trig_cellpose_cyto,
        multilabel=multilabel,
        aggregate_distance_factor=aggregate_distance_factor,
        pcm_thickness_factor=pcm_thickness_factor,
        size_reference=size_reference,
        iterative_split=iterative_split,
        oversize_factor=oversize_factor,
        max_split_iterations=max_split_iterations,
        merge_undersized=merge_undersized,
        undersize_factor=undersize_factor,
        split_gamma=split_gamma,
        split_gamma_step=split_gamma_step,
        min_piece_fraction=min_piece_fraction,
        sigma=sigma,
        num_plateaus=num_plateaus,
        plateau_factor=plateau_factor,
        threshold_method=threshold_method,
        export_vtk=export_vtk,
        export_stl=export_stl,
        export_fea=export_fea,
        settings=settings,
        napari_module=napari,
        progress=tqdm,
    )
    display(batch_summary)
    raise BatchCompleted("Batch processing finished -- the remaining cells are for single-file mode only.")
else:
    print("Single-file mode -- continue with the next cells.")

In [ ]:
annotate_gt = False  # True = paint ground-truth nuclei in napari (saved to ground_truth/)
annotator = ""       # name of the person annotating, saved in ground_truth/<image>.json
gt_draft = None      # None = paint from scratch, "cellpose" = correct a Cellpose draft
gt_restart = False   # True = ignore an existing ground_truth/<image>.tif and start again

if not annotate_gt:
    print("Ground-truth annotation skipped (annotate_gt = False).")
else:
    from validation.annotate_ground_truth import annotate_from_notebook
    # NUCLEI channel from the stain table, on the original voxel grid of the ROI
    # (before zoom): evaluate_segmentation.py maps validation_labels/ onto it.
    gt_path = annotate_from_notebook(
        img, input_file, ROI, file_meta, stain_dict,
        voxel_size=(r_Z, r_Y, r_X),
        annotator=annotator,
        out="ground_truth",
        draft=gt_draft,
        nuclei_diameter=nuclei_diameter,
        restart=gt_restart,
    )
    print(f"For evaluate_segmentation.py: --voxel-size {r_Z} {r_Y} {r_X}")

# 3. Preparation and Preview

**Cell 7** — Prepare the image stack and preview the channels in napari.

In [ ]:
(im_final_stack, nuclei_radius, cell_radius, nuclei_volume, cell_volume,
 stain_df, viewer_0) = prepare_and_preview(
    img,
    nuclei_diameter, cell_diameter,
    stain_dict, file_meta,
    napari_module=napari, r_xyz=(r_X, r_Y, r_Z), progress=tqdm,
)

**Cell 8** — Contrast and gamma per channel: reused from `<name_setup>_setup.csv`, set in napari, or automatic. The stain settings table is shown once the napari window is closed.

In [ ]:
stain_df, stain_complete_df, original_stain_complete_df = prepare_stain_settings(
    im_final_stack['Original image'],
    stain_df=stain_df,
    name_setup=name_setup,
    use_setup=use_setup,
    automatic_contrast=automatic_contrast,
    settings=settings,
    napari_module=napari,
    r_xyz=(r_X, r_Y, r_Z),
    progress=tqdm,
)
display(stain_complete_df)

# 4. Image Processing and Segmentation

Run in order. The parameters of Cells 13–15 are set in Cell 4.

**Cell 9** — Normalize each channel to 0–255.

In [ ]:
im_final_stack = run_normalize(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

**Cell 10** — Resample to isotropic voxels.

In [ ]:
im_final_stack, r_zX, r_zY, r_zZ = run_resample(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    zoom_factors=zooms,
    meta=meta,
)

**Cell 11** — Denoise (median filter).

In [ ]:
im_final_stack = run_denoise(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

**Cell 12** — Apply contrast and gamma from Cell 8.

In [ ]:
im_final_stack = run_contrast_gamma(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

**Cell 13** — Gaussian smoothing (`sigma`, Cell 4).

In [ ]:
im_final_stack = run_smooth(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    sigma=sigma,
)

**Cell 14** — Contrast enhancement (histogram equalization; `num_plateaus`, `plateau_factor`, Cell 4).

In [ ]:
im_final_stack = run_equalize(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    num_plateaus=num_plateaus,
    plateau_factor=plateau_factor,
)

**Cell 15** — Threshold every channel (global + local; `threshold_method`, Cell 4). The masks define nuclei, cytoplasm and marker positivity.

In [ ]:
im_final_stack = run_threshold(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    threshold_method=threshold_method,
    progress=tqdm,
)

**Cell 16** — Export histograms and parameters to `<file>_histograms.xlsx`.

In [ ]:
output_path = export_channel_histograms(
    im_final_stack, stain_complete_df, input_file,
    ROI_print=ROI_print, lazy_loading_used=used_lazy_loading,
    name_setup=name_setup,
    nuclei_diameter=nuclei_diameter, cell_diameter=cell_diameter,
    cyto_factor=cyto_factor, pcm_thickness_factor=pcm_thickness_factor,
    zoom_factors=zooms, scale_factor=scale_factor,
    trig_cellpose=trig_cellpose, trig_stardist=trig_stardist,
    trig_cellpose_cyto=trig_cellpose_cyto,
    multilabel=multilabel, cyto_markers=cyto_markers,
    membrane_markers=membrane_markers,
    sigma=sigma, num_plateaus=num_plateaus, plateau_factor=plateau_factor,
    threshold_method=threshold_method,
    aggregate_distance_factor=aggregate_distance_factor,
    extra_params={
        'Split gamma': split_gamma,
        'Split gamma step': split_gamma_step,
        'Min split piece fraction': min_piece_fraction,
    },
    progress=tqdm,
)

**Cell 17** — Segment nuclei with the chosen method, then refine by size (merge fragments, re-split oversized nuclei). The recap table gives each nucleus's final label.

In [ ]:
im_segmentation_stack = segment_nuclei(
    im_final_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    r_zxyz=(r_zX, r_zY, r_zZ),
    nuclei_diameter=nuclei_diameter,
    trig_stardist=trig_stardist,
    trig_cellpose=trig_cellpose,
    size_reference=size_reference,
    iterative_split=iterative_split,
    oversize_factor=oversize_factor,
    max_split_iterations=max_split_iterations,
    merge_undersized=merge_undersized,
    undersize_factor=undersize_factor,
    split_gamma=split_gamma,
    split_gamma_step=split_gamma_step,
    min_piece_fraction=min_piece_fraction,
    progress=tqdm,
)

In [ ]:
from validation.run_metadata import save_validation_labels
method_name = "full"  # e.g. "no_refinement", "split_gamma_off", "cellpose_plain"
# Every setting of Cells 3-4 (and the stain table, with contrast/gamma) goes to
# validation_labels/<method_name>/<image>.json, to reproduce and compare runs.
save_validation_labels(
    im_segmentation_stack["Nuclei"], input_file, method_name,
    roi=ROI,
    voxel_size=(r_zZ, r_zY, r_zX),
    stain_complete_df=stain_complete_df,
    settings={
        "name_setup": name_setup, "use_setup": use_setup,
        "automatic_contrast": automatic_contrast, "interactive_roi": interactive_roi,
        "stain_dict": stain_dict, "cyto_markers": cyto_markers,
        "membrane_markers": membrane_markers,
        "original_voxel_size_um_zyx": [r_Z, r_Y, r_X],
        "nuclei_diameter": nuclei_diameter, "cell_diameter": cell_diameter,
        "scale_factor": scale_factor, "zoom_factors": zoom_factors,
        "zooms_applied": zooms,
        "trig_cellpose": trig_cellpose, "trig_stardist": trig_stardist,
        "trig_cellpose_cyto": trig_cellpose_cyto,
        "multilabel": multilabel,
        "aggregate_distance_factor": aggregate_distance_factor,
        "pcm_thickness_factor": pcm_thickness_factor,
        "size_reference": size_reference, "iterative_split": iterative_split,
        "oversize_factor": oversize_factor, "max_split_iterations": max_split_iterations,
        "merge_undersized": merge_undersized, "undersize_factor": undersize_factor,
        "split_gamma": split_gamma, "split_gamma_step": split_gamma_step,
        "min_piece_fraction": min_piece_fraction,
        "sigma": sigma, "num_plateaus": num_plateaus, "plateau_factor": plateau_factor,
        "threshold_method": threshold_method,
    },
)

In [ ]:
preview_nuclei = True  # True = overlay EQ nuclei, threshold mask and segmented nuclei in napari

if not preview_nuclei:
    print("Nuclei preview skipped (preview_nuclei = False).")
else:
    nuc_c = stain_complete_df.index.get_loc("NUCLEI")
    scale_zoom = (r_zZ, r_zY, r_zX)
    viewer_nuc = napari.Viewer(title=f"Nuclei check - {method_name}")
    viewer_nuc.add_image(im_final_stack['Equalized image'][..., nuc_c], name="EQ NUCLEI",
                         colormap="gray", blending="additive", scale=scale_zoom)
    viewer_nuc.add_image((im_final_stack['Threshold image'][..., nuc_c] > 0).astype("uint8"),
                         name="Threshold NUCLEI", colormap="green", blending="additive",
                         contrast_limits=(0, 1), opacity=0.3, scale=scale_zoom)
    seg_layer = viewer_nuc.add_labels(im_segmentation_stack["Nuclei"], name="Segmented nuclei",
                                      opacity=0.6, scale=scale_zoom)
    seg_layer.contour = 1  # outlines only, so EQ and threshold stay visible inside
    print(f"{int(im_segmentation_stack['Nuclei'].max())} segmented nuclei.")

**Cell 18** — Segment cytoplasm: thresholded cytoplasm/membrane stains assigned to nuclei, one cell per nucleus. Nuclei without cytoplasm are excluded from marker analysis.

In [ ]:
im_segmentation_stack, stain_complete_df = segment_cytoplasm(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    cyto_markers=cyto_markers,
    membrane_markers=membrane_markers,
    cyto_factor=cyto_factor,
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    trig_cellpose_cyto=trig_cellpose_cyto,
    r_zxyz=(r_zX, r_zY, r_zZ),
    split_gamma=split_gamma,
    progress=tqdm,
)

**Cell 19** — Segment the pericellular matrix (PCM): a shell `pcm_thickness_factor` × `cell_diameter` thick around each cell.

In [ ]:
im_segmentation_stack = segment_pcm(
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    cell_diameter=cell_diameter,
    r_zxyz=(r_zX, r_zY, r_zZ),
    pcm_thickness_factor=pcm_thickness_factor,
)

**Cell 20** — Assign cells to the markers they are positive for.

In [ ]:
im_segmentation_stack = assign_channel_labels(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    progress=tqdm,
)

**Cell 21** — Detect cell aggregates: cells closer than `aggregate_distance_factor` × `cell_diameter` form one aggregate.

In [ ]:
im_segmentation_stack = detect_aggregates(
    im_segmentation_stack,
    stain_df=stain_df,
    cell_diameter=cell_diameter,
    r_zxyz=(r_zX, r_zY, r_zZ),
    aggregate_distance_factor=aggregate_distance_factor,
)

**Cell 22** — View processing steps and segmentation in napari.

In [ ]:
viewer_0, viewer_1 = view_processing_results(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    r_xyz=(r_X, r_Y, r_Z),
    r_zxyz=(r_zX, r_zY, r_zZ),
    napari_module=napari,
    progress=tqdm,
)

# 5. Quantification and Analysis

Intensities are reported for **positive cells** and **all cells**.

**Cell 23** — Measure intensity, size and position of every cell (`labels_df`).

In [ ]:
percell_mean_df, percell_std_df = compute_percell_marker_intensity_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    progress=tqdm,
)

labels_df = build_labels_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    zooms=zooms,
    multilabel=multilabel,
    percell_mean_df=percell_mean_df,
    percell_std_df=percell_std_df,
    progress=tqdm,
    is_2d=is_2d,
)

**Cell 24** — Population summary per condition.

In [ ]:
print_population_summary(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    progress=tqdm,
)

**Cell 25** — Full-resolution table for export.

In [ ]:
labels_full_df = build_full_labels_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    zooms=zooms,
    percell_mean_df=percell_mean_df,
    percell_std_df=percell_std_df,
    progress=tqdm,
    is_2d=is_2d,
)

**Cell 26** — Per-nucleus intensity report (PDF).

In [ ]:
hist_data, intensity_ranges, fig, axes, x_grid = build_histogram_report(
    im_segmentation_stack, im_final_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    show_plot=False,  # KDE plots go to the PDF only
    progress=tqdm,
)

**Cell 27** — Spatial distribution: cell counts (lines) and marker intensity (dots) along X, Y, Z.

In [ ]:
plot_spatial_distributions(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    im_final_stack=im_final_stack,
    r_X=r_X, r_Y=r_Y, r_Z=r_Z,
    zoom_factors=zooms,
    percell_mean_df=percell_mean_df,
    progress=tqdm,
)

**Cell 28** — Size distribution of nuclei and cells.

In [ ]:
plot_size_distributions(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    progress=tqdm,
)

**Cell 29** — Marker intensity vs cytoplasm size (flow-cytometry style).

In [ ]:
plot_marker_intensity_clouds(
    percell_mean_df,
    stain_complete_df=stain_complete_df,
    im_segmentation_stack=im_segmentation_stack,
    r_xyz=(r_zX, r_zY, r_zZ),
    stain_df=stain_df,
    progress=tqdm,
    input_file=input_file,
    is_2d=is_2d,
)

# 6. Export

3D exports are skipped for 2D images. Which ones run is set in Cell 4; the single-cell export is set in Cell 32.

**Cell 30** — VTK volumes for ParaView (`export_vtk`).

In [ ]:
if not export_vtk:
    print("VTK volumes skipped (export_vtk = False).")
elif is_2d:
    print("2D image: VTK volumes skipped (3D only).")
else:
    build_vtk_volumes(
        im_segmentation_stack,
        labels_full_df=labels_full_df,
        stain_complete_df=stain_complete_df,
        input_file=input_file,
        r_xyz=(r_X, r_Y, r_Z),
        zoom_factors=zooms,
        progress=tqdm,
    )

**Cell 31** — STL mesh per marker (`export_stl`).

In [ ]:
if not export_stl:
    print("STL meshes skipped (export_stl = False).")
elif is_2d:
    print("2D image: STL meshes skipped (3D only).")
else:
    export_marker_stl(
        im_segmentation_stack,
        stain_df=stain_df,
        stain_complete_df=stain_complete_df,
        input_file=input_file,
        r_xyz=(r_X, r_Y, r_Z),
        zoom_factors=zooms,
        progress=tqdm,
    )

**Cell 32** — Single-cell VTK export. Set `export_nuc_3D_vtk = True` in this cell and run it; it asks for the nucleus label (see the recap table of Cell 17).

In [ ]:
export_nuc_3D_vtk = False  # True = export one cell as VTK, asks for its label

if not export_nuc_3D_vtk:
    print("Single-cell VTK export skipped (export_nuc_3D_vtk = False).")
elif is_2d:
    print("2D image: single-cell VTK export skipped (3D only).")
else:
    nuc_label = int(input("Enter nucleus label number to export: "))
    export_nucleus_vtk_crop(
        nuc_label=nuc_label,
        im_segmentation_stack=im_segmentation_stack,
        im_final_stack=im_final_stack,
        stain_df=stain_df,
        input_file=input_file,
        r_zxyz=(r_zX, r_zY, r_zZ),
        zoom_factors=zooms,
    )

**Cell 33** — Excel report (`<file>_segmentation.xlsx`).

In [ ]:
output_path = export_quantification_to_excel(
    input_file,
    original_stain_complete_df,
    labels_full_df,
    progress=tqdm,
)

**Cell 34** — FEA mesh (Abaqus `.inp`; `export_fea`).

In [ ]:
if not export_fea:
    print("FEA mesh skipped (export_fea = False).")
elif is_2d:
    print("2D image: FEA mesh skipped (3D only).")
else:
    export_fea_mesh(
        im_segmentation_stack,
        input_file=input_file,
        r_zxyz=(r_zX, r_zY, r_zZ),
        progress=tqdm,
    )